In [6]:
import pandas as pd

# ===============================
# INPUTS
# ===============================
rep_file = "representatives_by_cluster_level.tsv"
cl_file  = "cluster_assignments_by_level.tsv"

# ===============================
# LOAD
# ===============================
rep = pd.read_csv(rep_file, sep="\t")
cl  = pd.read_csv(cl_file, sep="\t")

# ===============================
# STANDARDIZE COLUMN NAMES
# ===============================
rep = rep.rename(columns={
    "level": "cluster_level",
    "cluster": "cluster_id",
    "representative": "plasmid_id"
})

cl = cl.rename(columns={
    "level": "cluster_level",
    "cluster": "cluster_id",
    "id": "plasmid_id"
})

# ===============================
# RECOMPUTE CLUSTER SIZE
# ===============================
cluster_sizes = (
    cl.groupby(["cluster_level", "cluster_id"], dropna=False)
      .size()
      .reset_index(name="cluster_size")
)

# ===============================
# SHEET 1: SELECTED REPRESENTATIVES
# ===============================
selected_representatives = rep.merge(
    cluster_sizes,
    on=["cluster_level", "cluster_id"],
    how="left"
)

# si ya existe cluster_size en rep, priorizar esa;
# si falta, usar la recalculada
if "cluster_size_x" in selected_representatives.columns and "cluster_size_y" in selected_representatives.columns:
    selected_representatives["cluster_size"] = (
        selected_representatives["cluster_size_x"]
        .fillna(selected_representatives["cluster_size_y"])
    )
elif "cluster_size" not in selected_representatives.columns:
    # caso normal: solo entra la recalculada
    pass

# columnas finales
cols_selected = ["plasmid_id", "cluster_level", "cluster_id", "cluster_size"]
if "rep_size_metric" in selected_representatives.columns:
    cols_selected.append("rep_size_metric")

selected_representatives = (
    selected_representatives[cols_selected]
    .drop_duplicates()
    .sort_values(["cluster_level", "cluster_size", "cluster_id", "plasmid_id"],
                 ascending=[True, False, True, True])
    .reset_index(drop=True)
)

# ===============================
# SHEET 2: ALL PLASMIDS WITH CLUSTER ASSIGNMENT
# ===============================
all_plasmids = cl.merge(
    cluster_sizes,
    on=["cluster_level", "cluster_id"],
    how="left"
)

# marcar representantes
rep_flag = rep[["plasmid_id", "cluster_level", "cluster_id"]].drop_duplicates().copy()
rep_flag["is_representative"] = True

all_plasmids = all_plasmids.merge(
    rep_flag,
    on=["plasmid_id", "cluster_level", "cluster_id"],
    how="left"
)

# normalizar a booleano real
all_plasmids["is_representative"] = (
    all_plasmids["is_representative"]
    .replace({1: True, 0: False, "1": True, "0": False})
    .fillna(False)
    .astype(bool)
)

# si prefieres texto en vez de booleanos, descomenta esta línea:
all_plasmids["is_representative"] = all_plasmids["is_representative"].map({True: "Yes", False: "No"})

all_plasmids = (
    all_plasmids[
        ["plasmid_id", "cluster_level", "cluster_id", "cluster_size", "is_representative"]
    ]
    .drop_duplicates()
    .sort_values(["cluster_level", "cluster_id", "plasmid_id"],
                 ascending=[True, True, True])
    .reset_index(drop=True)
)

# ===============================
# EXPORT TO ONE EXCEL FILE WITH TWO SHEETS
# ===============================
out = "Supplementary_Dataset_representative_plasmids.xlsx"

with pd.ExcelWriter(out, engine="openpyxl") as writer:
    selected_representatives.to_excel(writer, sheet_name="selected_representatives", index=False)
    all_plasmids.to_excel(writer, sheet_name="all_plasmids_clustered", index=False)

print("Saved:", out)
print("selected_representatives:", selected_representatives.shape)
print("all_plasmids_clustered:", all_plasmids.shape)
print()
print(selected_representatives.head())
print()
print(all_plasmids.head())

/tmp/ipykernel_1823457/144475689.py:95: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


Saved: Supplementary_Dataset_representative_plasmids.xlsx
selected_representatives: (23925, 5)
all_plasmids_clustered: (53515, 5)

      plasmid_id  cluster_level  cluster_id  cluster_size  rep_size_metric
0  NZ_CP141563.1              1         572           521            22756
1     CP084538.1              1         633           362            26352
2  NZ_CP016550.1              1         259           270            13281
3  NZ_CP082017.1              1         621           263             2516
4  NZ_OZ040272.1              1         254           203              472

   plasmid_id  cluster_level  cluster_id  cluster_size is_representative
0  AB011548.2              1           0            58                No
1  CP043013.1              1           0            58                No
2  CP051825.1              1           0            58                No
3  CP051828.1              1           0            58                No
4  CP051831.1              1           0            5